# Data Preparation Pipeline of Salary Prediction

# Before Data Preparation, you must:
**1. Define the Research/Problem Objective**
- Clearly state what you want the data to help you achieve.
- Define the prediction, classification, detection, or analysis task.
  
**2. Define the Data Requirements**
- Determine what type of data you need.
- Identify required variables/features, labels, classes, metadata, etc.
- Define the expected data format and quality requirements.
  
**3. Identify Data Sources**
- Determine where the data will come from.
**Examples:** surveys, databases, sensors, cameras, public datasets, Roboflow, institutional records, or experiments.
  
**Collect/Acquire the Raw Data**
- Obtain the original data before applying transformations.
- Keep an untouched copy of the raw dataset.

### Why data preparation matters

Raw data is almost never ready to feed straight into a model. It usually contains duplicate records, missing values, outliers, skewed distributions, and features on wildly different scales — all of which can silently bias or break a model. **Data preparation is the process of systematically detecting and fixing these issues** so that the model learns real patterns instead of noise or artifacts.

**Pipeline overview:**

| # | Stage | Purpose |
|---|-------|---------|
| 1 | Data Collection & Integration | Bring raw data from its source(s) into one working dataset |
| 2 | Exploratory Data Analysis | Diagnose the data's quality and structure before changing anything |
| 3 | Data Cleaning | Fix dtypes, remove duplicates, impute missing values, remove outliers |
| 4 | Feature Transformations | Correct skewed distributions so patterns aren't dominated by extreme values |
| 5 | Categorical Encoding | Convert text categories into a numeric form models can use |
| 6 | Feature Engineering & Selection | Create potentially useful features, then keep only what earns its place |




## Step 1-Data Collection & Integration

Before any analysis or cleaning can happen, the data has to exist as a single, unified table in memory. In a real project this step might mean pulling from a database, calling an API, or merging several files together-here it's a single CSV, but the same principle applies: **consolidate first, so every later step has one consistent source of truth to work from.**

In [63]:
import pandas as pd  # pandas gives us the DataFrame structure used throughout the pipeline
# Load the CSV file into a DataFrame
# (in a real project this could instead be a SQL query, an API call, or several files merged together)
df = pd.read_csv("Salary_dataset.csv")

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")  # how many records (rows) and variables (columns) are in my dataset,confirm the data loaded with the expected dimensions
print(f"Column names: {list(df.columns)}")             # confirm the expected columns are present
df.head()  # visually spot-check the first few rows

Rows: 70, Columns: 2
Column names: ['YearsExperience', 'Salary']


,YearsExperience,Salary
0,1.2,39344.0
1,1.4,46206.0
2,1.6,37732.0
3,NaN,43526.0
4,2.3,39892.0


## Step 2-Exploratory Data Analysis (EDA)

You can't fix problems you haven't found. EDA is a **purely diagnostic** stage-nothing is changed here. The goal is to build an accurate mental model of the data's quality: its statistical shape, its data types, where values are missing, whether rows are duplicated, and how many distinct values each column holds. Everything decided in Step 3 (cleaning) is based on what's discovered here.

In [64]:
print("--- 2.1 Basic statistics ---")
display(df.describe())  # count, mean, std, min/max, quartiles — gives an immediate sense of scale and spread

--- 2.1 Basic statistics ---


,YearsExperience,Salary
count,64.000000,67.000000
mean,5.300000,73702.298507
std,2.770823,26208.857961
min,1.200000,37732.000000
25%,3.300000,56643.000000
50%,4.100000,63219.000000
75%,7.400000,96107.500000
max,10.600000,122392.000000


In [65]:
print("---2.2 Data types---")
print(df.dtypes)  # confirms whether pandas parsed each column as numeric, text, etc. — wrong dtypes cause silent bugs later

---2.2 Data types---
YearsExperience    float64
Salary             float64
dtype: object


In [66]:
print("---2.3 Missing values per column---")
print(df.isna().sum())  # count of NaN/missing entries per column — tells us whether imputation will be needed

---2.3 Missing values per column---
YearsExperience    6
Salary             3
dtype: int64


In [67]:
print("--- 2.4 Duplicate rows ---")
print(f"Duplicate rows: {df.duplicated().sum()}")  # exact duplicate rows would otherwise double-count observations and bias the model

--- 2.4 Duplicate rows ---
Duplicate rows: 32


In [68]:
print("--- 2.5 Unique values ---")
for col in df.columns:               # loop over every column
    print(f"{col}: {df[col].nunique()} unique values")  # cardinality check — helps decide later whether a column is categorical or continuous

--- 2.5 Unique values ---
YearsExperience: 29 unique values
Salary: 31 unique values


**Findings:** the dataset has missing values in both columns and a meaningful number of duplicate rows — both need to be addressed in the cleaning step. No changes are made yet; this stage is purely diagnostic.

## Step 3-Data Cleaning

This is where the problems found in EDA actually get fixed. Cleaning matters because models trained on dirty data learn the noise, not the signal: duplicates cause certain observations to be over-weighted, missing values crash many algorithms outright, and outliers can dominate a model's fitted parameters (especially for linear models, which are sensitive to extreme values).

In [69]:
# --- 3.1 Fix data types ---
# Purpose: guarantee both columns are truly numeric. Any value that can't be converted
# (e.g. a stray string like "N/A") becomes NaN instead of silently breaking downstream math.
df["YearsExperience"] = pd.to_numeric(df["YearsExperience"], errors="coerce")  # coerce = invalid values become NaN rather than raising an error
df["Salary"] = pd.to_numeric(df["Salary"], errors="coerce")  # coerce =If a value cannot be converted to a number, don't stop the program. Change that value to NaN instead.
print("3.1 Data types coerced to numeric")

3.1 Data types coerced to numeric


In [70]:
# --- 3.2 Remove duplicates ---
before = len(df)                # counts the total number of rows in your DataFrame
df = df.drop_duplicates()       # keeps the first occurrence of each duplicated row, drops the rest(looks for rows that are exactly the same and removes the repeated copies.
print(f"3.2 Removed {before - len(df)} duplicate rows") #Calculate how many were removed

3.2 Removed 32 duplicate rows


In [71]:
# --- 3.3 Handle missing values (imputation) ---
# Purpose: most ML algorithms cannot handle NaN directly, so missing values must be filled in
# with a reasonable estimate rather than dropped (which would throw away otherwise-good data).
missing=df.isna().sum().sum()  # total count of missing cells, for reporting
df["YearsExperience"] = df["YearsExperience"].fillna(df["YearsExperience"].median())  # median: robust to outliers
df["Salary"] = df["Salary"].fillna(df["Salary"].mean())                               # mean: fine here since Salary isn't heavily skewed
print(f"3.3 Filled {missing} missing values (median for YearsExperience, mean for Salary)")

3.3 Filled 9 missing values (median for YearsExperience, mean for Salary)


**3.4 Outlier detection using the IQR(Interquartile Range) method.** An outlier is a value that is unusually far from most of the other values in a dataset.
Extreme values (e.g. a data-entry typo of 500 years of experience) can distort a model far more than their single data point should. The IQR method flags anything unusually far from the bulk of the distribution using the 25th/75th percentiles, rather than an arbitrary fixed cutoff.

In [72]:
def remove_outliers(data, col):
    Q1 = data[col].quantile(0.25)       # 25th percentile — the "lower edge" of the typical range
    Q3 = data[col].quantile(0.75)       # 75th percentile — the "upper edge" of the typical range
    IQR = Q3 - Q1                       # interquartile range: the spread of the middle 50% of the data
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR  # standard 1.5x IQR fences used to flag outliers
    before = len(data)                  # row count before filtering, for reporting
    data = data[(data[col] >= lower) & (data[col] <= upper)]  # keep only rows within the acceptable range
    print(f"   {col}: removed {before - len(data)} outliers (bounds {lower:.1f} to {upper:.1f})")
    return data

print("3.4 Outlier removal (IQR):")
df = remove_outliers(df, "YearsExperience")  # apply outlier filtering to each numeric column independently
df = remove_outliers(df, "Salary")

print(f"\nFinal shape after cleaning: {df.shape}")  # sanity check on how much data survived cleaning

3.4 Outlier removal (IQR):
   YearsExperience: removed 0 outliers (bounds -2.1 to 12.7)
   Salary: removed 0 outliers (bounds -3981.8 to 157894.2)

Final shape after cleaning: (38, 2)


In [73]:
df.to_csv("cleaned_dataset.csv", index=False)  # checkpoint the cleaned dataset
print("cleaned_dataset.csv")

PermissionError: [Errno 13] Permission denied: 'cleaned_dataset.csv'

## Step 4-Feature Transformations

Many machine learning models work better when the data is balanced and on similar scales.If a feature has a few very large values, the data can become skewed. These large values may affect the model too much.A log transformation can reduce the effect of very large values and make the data more balanced.Scaling means changing the values so that different features are on a similar range. Skewness is a measure that tells us how much a dataset is tilted or stretched toward the left or right.

In [41]:
df = pd.read_csv("cleaned_dataset.csv") 
print("4.1 Skewness (|skew| > 1 is considered highly skewed):")
print(df.skew())  # skew() measures distribution asymmetry: 0 = symmetric, positive = long right tail, negative = long left tail

4.1 Skewness (|skew| > 1 is considered highly skewed):
YearsExperience    0.547819
Salary             0.400081
dtype: float64


In [42]:
# --- 4.2 Log-transform any highly skewed column ---
# Purpose: log1p compresses large values much more than small ones, pulling in long tails
# and making a skewed distribution closer to symmetric (which most models handle better).
for col in df.columns:                    # check every numeric column
    if abs(df[col].skew()) > 1:           # only transform columns that are actually skewed
        print(f"   Applying log transform to {col}")
        df[col] = np.log1p(df[col])       # log1p(x) = log(1 + x), safely handles values at/near 0
    else:
        print(f"   {col}: skew OK, no transform needed")

   YearsExperience: skew OK, no transform needed
   Salary: skew OK, no transform needed


## Step 5-Categorical Encoding

Machine learning models operate on numbers, not text. Any categorical column (e.g. a job title or department name) has to be converted into a numeric representation before it can be used. **One-hot encoding** is a common, safe default: it creates a separate 0/1 column for each category, avoiding the trap of implying a false numeric order (e.g. accidentally telling the model that "Male" < "Female" if you just assigned 0/1 labels to a single column).

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")   # start from the Step 4 checkpoint

cat_cols = df.select_dtypes(include="object").columns.tolist()  # find columns holding text/object dtype

if cat_cols:  # only encode if categorical columns actually exist
    print(f"Categorical columns found: {cat_cols}")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)  # Convert categorical/text columns into 0 and 1 so they can be used by a machine learning model, and remove one category from each variable to avoid redundancy.
    print("Applied one-hot encoding")
else:
    print("No categorical columns — nothing to encode.")
    print("Both columns (YearsExperience, Salary) are already numeric.")

print(f"\nFinal shape: {df.shape}")  # shape often grows here if one-hot encoding added new columns

## Step 6-Feature Engineering & Selection
The original data may not show all the patterns needed by the machine learning model. **Feature engineering** creates new columns from existing data to help the model find patterns. **Feature selection** then checks these new columns and keeps only the useful ones. Unnecessary features are removed because they can make the model more complex and less accurate.

In [ ]:
# Load the cleaned data
df = pd.read_csv("cleaned_dataset.csv")

# Create a new feature by squaring years of experience
df["ExperienceSquared"] = df["YearsExperience"]**2

# Create a new feature for senior employees
# 1 = 5 or more years of experience
# 0 = less than 5 years
df["IsSenior"] = (df["YearsExperience"] >= 5).astype(int)

# Show the table
print("\nFeature Engineering Results:")
print(df[["YearsExperience", "ExperienceSquared", "IsSenior"]])

# Save the new dataset
df.to_csv("feature_engineered_dataset.csv", index=False)

## Next step is to train a regression model